In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [25]:
df = pd.read_csv('taxi_trip_pricing.csv')

In [26]:
df.head()

,Trip_Distance_km,Time_of_Day,Day_of_Week,Passenger_Count,Traffic_Conditions,Weather,Base_Fare,Per_Km_Rate,Per_Minute_Rate,Trip_Duration_Minutes,Trip_Price
0,19.35,Morning,Weekday,3.0,Low,Clear,3.56,0.80,0.32,53.82,36.2624
1,47.59,Afternoon,Weekday,1.0,High,Clear,NaN,0.62,0.43,40.57,NaN
2,36.87,Evening,Weekend,1.0,High,Clear,2.70,1.21,0.15,37.27,52.9032
3,30.33,Evening,Weekday,4.0,Low,NaN,3.48,0.51,0.15,116.81,36.4698
4,NaN,Evening,Weekday,3.0,High,Clear,2.93,0.63,0.32,22.64,15.6180


In [27]:
df.columns

Index(['Trip_Distance_km', 'Time_of_Day', 'Day_of_Week', 'Passenger_Count',
       'Traffic_Conditions', 'Weather', 'Base_Fare', 'Per_Km_Rate',
       'Per_Minute_Rate', 'Trip_Duration_Minutes', 'Trip_Price'],
      dtype='object')

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Trip_Distance_km       950 non-null    float64
 1   Time_of_Day            950 non-null    object 
 2   Day_of_Week            950 non-null    object 
 3   Passenger_Count        950 non-null    float64
 4   Traffic_Conditions     950 non-null    object 
 5   Weather                950 non-null    object 
 6   Base_Fare              950 non-null    float64
 7   Per_Km_Rate            950 non-null    float64
 8   Per_Minute_Rate        950 non-null    float64
 9   Trip_Duration_Minutes  950 non-null    float64
 10  Trip_Price             951 non-null    float64
dtypes: float64(7), object(4)
memory usage: 86.1+ KB


In [29]:
df.isna().sum()

Trip_Distance_km         50
Time_of_Day              50
Day_of_Week              50
Passenger_Count          50
Traffic_Conditions       50
Weather                  50
Base_Fare                50
Per_Km_Rate              50
Per_Minute_Rate          50
Trip_Duration_Minutes    50
Trip_Price               49
dtype: int64

In [30]:
df.shape

(1000, 11)

In [31]:
df = df.dropna(subset=['Trip_Price'])

In [32]:
df['Trip_Price'].isna().sum()

np.int64(0)

In [33]:
df.isna().sum()

Trip_Distance_km         50
Time_of_Day              49
Day_of_Week              46
Passenger_Count          48
Traffic_Conditions       50
Weather                  46
Base_Fare                44
Per_Km_Rate              44
Per_Minute_Rate          49
Trip_Duration_Minutes    46
Trip_Price                0
dtype: int64

In [34]:
x = df.drop('Trip_Price',axis=1)
y = df['Trip_Price']

In [35]:
X_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
y_train

340     73.5818
571     72.9188
584     65.6944
115    106.0042
81      87.6076
         ...   
112     99.8038
291      8.9203
904     31.6941
461     55.8212
107     66.2817
Name: Trip_Price, Length: 760, dtype: float64

In [36]:
numerical_features = [
    'Trip_Distance_km',
    'Passenger_Count',
    'Base_Fare',
    'Per_Km_Rate',
    'Per_Minute_Rate',
    'Trip_Duration_Minutes'
]

categorical_feature = [
    'Time_of_Day',
    'Day_of_Week',
    'Traffic_Conditions',
    'Weather'
]

In [37]:
numerical_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='mean')),
    ('scaling',StandardScaler())
])

categorical_pipeline = Pipeline([
   ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoding',OneHotEncoder())
])


In [38]:
numerical_pipeline

Pipeline(steps=[('imputer', SimpleImputer()), ('scaling', StandardScaler())])

In [39]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num',numerical_pipeline,numerical_features),
        ('cat',categorical_pipeline,categorical_feature)
    ]
)

In [40]:
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaling',
                                                  StandardScaler())]),
                                 ['Trip_Distance_km', 'Passenger_Count',
                                  'Base_Fare', 'Per_Km_Rate', 'Per_Minute_Rate',
                                  'Trip_Duration_Minutes']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoding',
                                                  OneHotEncoder())]),
                                 ['Time_of_Day', 'Day_of_Week',
                                  'Traffic_Conditions', 'Weather'])])

In [41]:
svr = SVR(kernel='rbf',C=10)

In [42]:
final_workflow = Pipeline([
    ('col',preprocessor),
    ('model',svr)
])

In [43]:
final_workflow

Pipeline(steps=[('col',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaling',
                                                                   StandardScaler())]),
                                                  ['Trip_Distance_km',
                                                   'Passenger_Count',
                                                   'Base_Fare', 'Per_Km_Rate',
                                                   'Per_Minute_Rate',
                                                   'Trip_Duration_Minutes']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoding',
                                                                   OneHotEncoder())]),
                                                  ['Time_of_Day', 'Day_of_Week',
                                                   'Traffic_Conditions',
                                                   'Weather'])])),
                ('model', SVR(C=10))])

In [44]:
final_workflow.fit(X_train,y_train)

Pipeline(steps=[('col',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaling',
                                                                   StandardScaler())]),
                                                  ['Trip_Distance_km',
                                                   'Passenger_Count',
                                                   'Base_Fare', 'Per_Km_Rate',
                                                   'Per_Minute_Rate',
                                                   'Trip_Duration_Minutes']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoding',
                                                                   OneHotEncoder())]),
                                                  ['Time_of_Day', 'Day_of_Week',
                                                   'Traffic_Conditions',
                                                   'Weather'])])),
                ('model', SVR(C=10))])

In [47]:
pred = final_workflow.predict(x_test)

In [46]:
final_workflow.score(X_train,y_train)

0.6268133391563562

In [48]:
r2_score(y_test,pred)

0.4311258478319355

In [49]:
mean_absolute_error(y_test,pred)

8.885154465695372

In [50]:
mean_squared_error(y_test,pred)

1329.6744144344436